# Silhouette Coefficient: From First Principles

The Silhouette Coefficient is an internal clustering metric: it evaluates labels using the feature-space distances without needing known ground-truth classes. It combines two ideas for each observation: how close it is to its assigned cluster and how far it is from the nearest alternative cluster.

Because the result depends on the distance metric and feature representation, a silhouette score measures the quality of a partition under those choices. It does not prove that clusters are real, useful, or correct for a downstream purpose.

## The Main Idea

A good cluster should satisfy two conditions. Points inside the same cluster should be close to each other. Points in different clusters should be far from each other.

Silhouette Coefficient measures both ideas at the same time.

It asks: is this point closer to its own cluster than to the nearest other cluster?


## Within-Cluster Distance $a(i)

For observation $i$, $a(i)$ is the mean distance from $i$ to the other observations assigned to its own cluster. Smaller values mean the point is closer, on average, to its cluster-mates.

The point itself is excluded from this average. If a cluster contains only the point $i$, there are no other members and $a(i)$ is undefined; common implementations assign that point a silhouette of 0 by convention.

## Nearest-Other-Cluster Distance $b(i)

For each cluster other than the one containing $i$, calculate the mean distance from $i$ to every member of that cluster. $b(i)$ is the smallest of those means.

This is not simply the distance to the nearest individual point in another cluster. Averaging over each candidate cluster makes $b(i)$ measure how close the point is to its nearest alternative group. A larger $b(i)$ relative to $a(i)$ supports the current assignment.

## Formula and Distance Definitions

Let $C_i$ be the cluster containing observation $i$, and let $d(i,j)$ be the chosen distance between observations. The within-cluster distance is the average distance from $i$ to the *other* observations in its own cluster:

$$
a(i)=\frac{1}{|C_i|-1}\sum_{j\in C_i,\,j\ne i}d(i,j)
$$

For any other cluster $C$, compute the average distance from $i$ to its members. The nearest-other-cluster distance is the smallest of these averages:

$$
b(i)=\min_{C\ne C_i}\left[\frac{1}{|C|}\sum_{j\in C}d(i,j)\right]
$$

The silhouette coefficient for observation $i$ is:

$$
s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}
$$

When the denominator is positive, $s(i)$ lies between $-1$ and $1$. A singleton cluster has no other within-cluster observations, so implementations commonly define its silhouette as 0 by convention.

## Interpreting Individual and Average Scores

- A score near $+1$ means $b(i)$ is much larger than $a(i)$, so the point is compactly placed in its assigned cluster and separated from the nearest alternative.
- A score near 0 means the within-cluster and nearest-other-cluster distances are similar; the point lies near a boundary or between groups.
- A negative score means $a(i)>b(i)$, so the point is, on average, closer to another cluster than to the one assigned.

The overall silhouette score is the arithmetic mean across observations:

$$
S=\frac{1}{n}\sum_{i=1}^{n}s(i)
$$

Because this average is weighted by the number of observations in each cluster, a large cluster can dominate the score. Inspect per-cluster averages or a silhouette plot as well: they can reveal small, poorly separated, or uneven clusters that a single global mean hides.

## Worked Examples

### A Well-Matched Point

Suppose the one-dimensional clusters are $C_1=\{0,2\}$ and $C_2=\{8,10\}$. For point $i=0$, its average distance to the other point in its cluster is $a(i)=2$. Its average distance to the only other cluster is $b(i)=(8+10)/2=9$. Thus:

$$
s(i)=\frac{9-2}{\max(2,9)}=\frac{7}{9}\approx0.78
$$

This positive score indicates that the point is much closer to its own cluster than to the alternative cluster.

### A Possibly Misassigned Point

Now let $C_1=\{0,2,6\}$ and $C_2=\{8,10\}$. For point $i=6$ in $C_1$, $a(i)=(6+4)/2=5$, while the average distance to $C_2$ is $b(i)=(2+4)/2=3$. Therefore:

$$
s(i)=\frac{3-5}{\max(5,3)}=-0.4
$$

The negative score suggests that point 6 is, on average, closer to the other cluster than to its assigned cluster.

## Compare Two Candidate Partitions

Use the one-dimensional points $\{0,2,5,7\}$ and absolute distance.

### Candidate $K=2$: $\{0,2\}$ and $\{5,7\}$

For point 0, $a=2$ and $b=(5+7)/2=6$, so $s=2/3$. For point 2, $a=2$ and $b=(3+5)/2=4$, so $s=1/2$. The other two points have the same scores in reverse: $1/2$ and $2/3$. Each cluster's mean is $7/12\approx0.583$, and the overall mean is also $7/12\approx0.583$.

### Candidate $K=3$: $\{0,2\}$, $\{5\}$, and $\{7\}$

For point 0, $a=2$ and its nearest other-cluster average is $b=5$, giving $s=3/5$. For point 2, $a=2$ and $b=3$, giving $s=1/3$. The two singleton clusters receive score 0 by convention. The two-point cluster's mean is $7/15\approx0.467$; the singleton-cluster means are 0, so the overall mean is $7/30\approx0.233$.

Under this metric and distance, the $K=2$ partition scores higher. The example also shows why reporting only the overall score can hide the singleton clusters' behavior.

## Comparing Clusterings and the Elbow Method

For a fixed feature representation and distance metric, the average silhouette score can compare candidate clusterings, such as K-Means models with different $K$. A larger score indicates better average within-cluster cohesion relative to separation from the nearest alternative cluster under that metric.

The Elbow Method is different: it plots K-Means inertia as $K$ changes and looks for a bend where extra clusters give diminishing reductions in within-cluster sum of squares. Silhouette combines within-cluster and nearest-other-cluster distances; inertia measures only the centroid-based within-cluster objective.

Neither method determines the correct clustering by itself. Use the same preprocessing and distance metric when comparing candidates, inspect cluster-level results, and consider whether the groups are useful for the application. A score optimized across many candidate settings can also overstate quality; use independent validation or stability checks when needed.

## When to Use It and Limitations

Use the Silhouette Coefficient when cluster labels are available but ground-truth classes are not, and a distance-based internal check is appropriate. It is useful for comparing candidate clusterings and identifying points that may be poorly assigned.

The metric favors groups that are compact and separated under the chosen distance. It can undervalue valid non-convex clusters, respond poorly to clusters with very different densities or sizes, and become difficult to interpret in high-dimensional spaces where distances are less informative. A high average does not prove that a clustering is meaningful.

The standard calculation requires a valid partition with at least two and fewer than $n$ labels. It can be expensive because it relies on many pairwise distances; sampling can reduce the cost for large datasets, at the expense of approximation.

For density-based methods such as DBSCAN, noise points are often given a shared label such as $-1$. Treating all noise as one ordinary cluster can distort the score, so state how noise was handled and inspect the noise fraction separately. Interpret silhouette together with domain knowledge, cluster-level diagnostics, and stability.